In [6]:
import pandas as pd
import numpy as np


## Data extraction, total in tonnes

### Load Data

In [166]:
data = pd.read_csv('../data/external/env_wasgen_linear_2_0.csv')

In [167]:
#Loading waste category excel to aggregate the waste categories in the data
#categories = pd.read_excel('../data/processed/categories_msw_composition.xlsx', sheet_name = 'Sheet1') #excludes mineral waste and chemical waste 
categories = pd.read_excel('../data/processed/categories_msw_composition.xlsx', sheet_name = 'Sheet2')
categories = categories[['Aggregated categories', 'Aggregated category name']]
categories = categories.loc[categories['Aggregated categories'].notna()].reset_index()

### Choosing household waste data

In [168]:
#hhw = data.loc[(data['Statistical classification of economic activities in the European Community (NACE Rev. 2)'] == 'Households')].copy().reset_index()
hhw = data.loc[(data['Statistical classification of economic activities in the European Community (NACE Rev. 2)'] == 'Households')&(data['hazard']=='HAZ_NHAZ')].copy().reset_index()

In [169]:
categories_map = {
    'W01-05': ['W011', 'W012','W013', 'W02A', 'W032', 'W033', 'W05'],
    'W06':['W061', 'W062','W063'],
    'W071':['W071'],
    'W072':['W072'],
    'W073': ['W073'],
    'W074':['W074'],
    'W075': ['W075'],
    'W076':['W076'],
    'W077_08':['W077',  'W081', 'W0841', 'W08A'],
    'W09':['W091', 'W092', 'W093'],
  #  'W10':['W101', 'W102', 'W103'],
    'W10':['W101', 'W102'], #removing sorting residues
    'W11':['W11'],
    'W12-13':[ 'W121', 'W124','W126', 'W127', 'W128_13']

}

In [170]:
reverse_map = {}
for category_key, waste_codes in categories_map.items():
    for waste_code in waste_codes:
        reverse_map[waste_code] = category_key

In [171]:
hhw=hhw[['unit','waste', 'Waste categories', 'geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD',  'OBS_VALUE']]

In [172]:
hhw['category code'] = hhw['waste'].map(reverse_map)

In [173]:
hhw = hhw.loc[~hhw['category code'].isna()]


In [174]:
hhw_cat = hhw.groupby(['unit', 'geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD', 'category code'], as_index=False)['OBS_VALUE'].sum()


In [175]:
del data

In [176]:
#Test
hhw_cat.loc[(hhw_cat['geo']=='IT')&(hhw_cat['category code'].isin(['W10', 'W101', 'W102', '103']))&(hhw_cat['unit']=='T')]

,unit,geo,Geopolitical entity (reporting),TIME_PERIOD,category code,OBS_VALUE
6762,T,IT,Italy,2004,W10,24376698.0
6774,T,IT,Italy,2006,W10,24464573.0
6786,T,IT,Italy,2008,W10,22838700.0
6799,T,IT,Italy,2010,W10,21423512.0
6812,T,IT,Italy,2012,W10,18458498.0
6825,T,IT,Italy,2014,W10,16796295.0
6838,T,IT,Italy,2016,W10,15535099.0
6851,T,IT,Italy,2018,W10,14139156.0
6864,T,IT,Italy,2020,W10,12353544.0
6877,T,IT,Italy,2022,W10,11946906.0


In [ ]:
#hhw = hhw.loc[hhw['waste'].isin(categories['Aggregated categories'].unique())].copy().reset_index(drop=True)

In [177]:
hhw_cat= pd.merge(hhw_cat, categories[['Aggregated categories', 'Aggregated category name']], left_on ='category code', right_on = 'Aggregated categories', how = 'left').drop(columns =['Aggregated categories'])

In [178]:
#Getting the volume data (in tonnes)
hhw_t = hhw_cat.loc[hhw_cat['unit']=='T'].copy().reset_index(drop=True)

### Summing WEEE and battery waste categories

In [61]:
weee_batt_cat = ['W077', 'W0841', 'W08A']
weee = hhw_t.loc[hhw_t['waste'].isin(weee_batt_cat)].copy().reset_index(drop=True)


In [62]:
weee = weee.groupby(['unit', 'Unit of measure', 'geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD'], as_index=False)['OBS_VALUE'].sum()


In [63]:
weee['waste'] = 'W07_08'
weee['Waste categories'] = 'weee + batt, no elv'

In [64]:
hhw_t = hhw_t.loc[~hhw_t['waste'].isin(weee_batt_cat)].copy()
hhw_t = pd.concat([hhw_t, weee], ignore_index=True)

In [179]:
hhw_t['Aggregated category name'].unique()

array(['mixed waste', 'chemical and medical', 'glass', 'paper', 'rubber',
       'plastic', 'wood', 'textile', 'weee + batt + vehicles',
       'organic waste', 'common sludges', 'mineral waste', 'metal'],
      dtype=object)

### Getting the sum of textile and rubber wastes

In [180]:
textile = hhw_t.loc[hhw_t['Aggregated category name'].isin(['textile','rubber'])].copy().reset_index(drop=True)
textile = textile.groupby(['unit', 'geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD'], as_index=False)['OBS_VALUE'].sum()
textile['Aggregated category name'] = 'textile + rubber'
textile['category code'] = 'W073_076'


In [181]:
hhw_t = pd.concat([hhw_t.loc[~hhw_t['Aggregated category name'].isin(['textile', 'rubber'])], textile], ignore_index=True )

In [182]:
country_codes = pd.read_csv('../data/processed/country_codes.csv')

In [183]:
hhw_t = pd.merge(hhw_t, country_codes[['Code', 'NUTS_code']], left_on='geo', right_on='Code', how='left').drop(columns=['Code', 'geo'])

In [184]:
hhw_t.to_csv('../data/processed/msw_wf_composition_v4.csv', index=False)
#v1 doesnot include all waste categories, and does not select only totals (HAZ, NHAZ)
#v2 adjusts for this but the totals are not checked
#v3 totals are calcualted instead of used preestimated sums
#v4 excludes sorting residue in sum

## Data extraction, percentages

In [73]:
hhw_t = hhw_t.pivot_table(index = ['unit','NUTS_code', 'Geopolitical entity (reporting)',
       'TIME_PERIOD'], columns = 'Waste categories', values='OBS_VALUE').reset_index()

In [74]:
id_cols = ['unit','NUTS_code','Geopolitical entity (reporting)','TIME_PERIOD']
cat_cols = hhw_t.columns.difference(id_cols)

In [75]:
hhw_percent = hhw_t.copy()
hhw_percent[cat_cols] = hhw_t[cat_cols].div(
    hhw_t[cat_cols].sum(axis=1).replace(0, np.nan), axis=0
) * 100

In [76]:
hhw_percent.to_csv('../data/processed/msw_wf_composition_percent.csv', index=False)